第七课

Use AI (机器学习 / 深度学习: 随机森林、Gradient Boosting Regressor、长短期记忆网络) to predict future cash flows

Use DCF -> 估值

In [36]:
import rqdatac as rq
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.metrics import r2_score, mean_absolute_error

# Log in to rqdatac
rq.init()

# Get 沪深300 constituent stocks
constituents = rq.index_components('000300.XSHG')

start_date = '2010-01-01'
end_date = '2025-12-31'

factors = [
    # revenue
    'revenue',
    'operating_revenue',
    'net_operating_revenue',
    
    # valuation
    'book_value_per_share',
    'book_value_per_share_lf',
    'book_value_per_share_ttm',

    # profitability
    'return_on_equity',
    'return_on_asset',
    "profit_before_tax",
    "net_profit",
    "net_profit_parent_company",
    "ebit",

    # leverage/liquidity
    'debt_to_equity',
    'current_ratio',
    
    # growth
    'revenue_growth_rate',
    'net_profit_growth_rate',

    # free cash flow
    'free_cash_flow_company_per_share',
    'free_cash_flow_company_per_share_ttm',

    # working capital
    "working_capital",
    "working_capital_lf",
    "working_capital_ttm",

    # investment
    "net_current_investment",

    # assets and liabilities
    "current_assets",
    "current_liabilities",

    "depreciation_and_amortization",

    "ebitda",

    # Balance
    "cash_equivalent", 
    "bill_receivable", 
    "net_accts_receivable", 
    "inventory", 
    "long_term_equity_investment", 
    "net_long_term_equity_investment", 
    "net_fixed_assets", 
    "intangible_assets", 
    "short_term_loans", 
    "long_term_liabilities_due_one_year", 
    "long_term_loans", 
    "bond_payable", 
    "long_term_payable", 
    "total_assets", 
    "equity_parent_company", 
    "total_equity",
]

target = 'free_cash_flow_company_per_share_ttm'

# Fetch financial statement ratio information
financial_data = rq.get_factor(constituents, factors, start_date=start_date, end_date=end_date)

financial_data = financial_data.fillna(0, inplace=False)

print("financial data:")
print(financial_data.head(5))

/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.8/site-packages/rqdatac/client.py:224: UserWarning: rqdatac is already inited. Settings will be changed.
  warnings.warn("rqdatac is already inited. Settings will be changed.", stacklevel=0)


financial data:
                               revenue  operating_revenue  \
order_book_id date                                          
600674.XSHG   2010-01-04  6.628151e+08       6.628151e+08   
              2010-01-05  6.628151e+08       6.628151e+08   
              2010-01-06  6.628151e+08       6.628151e+08   
              2010-01-07  6.628151e+08       6.628151e+08   
              2010-01-08  6.628151e+08       6.628151e+08   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
600674.XSHG   2010-01-04                    0.0                  2.99   
              2010-01-05                    0.0                  2.99   
              2010-01-06                    0.0                  2.99   
              2010-01-07                    0.0                  2.99   
              2010-01-08                    0.0                  2.99   

                          book_value_per_sha

In [37]:
X = financial_data

X = X.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

X = X.reorder_levels(['order_book_id', 'date'])

print(X.head(5))

                               revenue  operating_revenue  \
order_book_id date                                          
000001.XSHE   2010-12-31  1.315964e+10       1.315964e+10   
              2011-12-31  2.070140e+10       2.070140e+10   
              2012-12-31  2.953167e+10       2.953167e+10   
              2013-12-31  3.734500e+10       3.734500e+10   
              2014-12-31  5.465100e+10       5.465100e+10   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
000001.XSHE   2010-12-31                    0.0                  9.22   
              2011-12-31                    0.0                 13.61   
              2012-12-31                    0.0                 15.95   
              2013-12-31                    0.0                 11.58   
              2014-12-31                    0.0                 11.09   

                          book_value_per_share_lf  book_valu

In [38]:
# Fetch next 5 years free cash flow
future_free_cash_flow = rq.get_factor(constituents, target, start_date='2015-01-01', end_date=end_date)

future_free_cash_flow = future_free_cash_flow.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

future_free_cash_flow = future_free_cash_flow.reorder_levels(['order_book_id', 'date'])

future_free_cash_flow = future_free_cash_flow.fillna(0, inplace=False)

y = future_free_cash_flow

print(future_free_cash_flow)

                          free_cash_flow_company_per_share_ttm
order_book_id date                                            
000001.XSHE   2015-12-31                              1.859431
              2016-12-31                              1.384475
              2017-12-31                              0.981718
              2018-12-31                              1.520333
              2019-12-31                              0.805708
...                                                        ...
688981.XSHG   2021-12-31                              0.648618
              2022-12-31                              1.428541
              2023-12-31                              1.360557
              2024-12-31                              0.078630
              2025-12-31                             -0.462069

[2927 rows x 1 columns]


In [39]:
# Assuming your data is already multi-indexed with (order_book_id, date)
# Align features (year t) with target (year t+1)
import numpy as np

# Method 1: Shift per stock
def align_features_target(X, y):
    """
    X: features at time t
    y: target at time t
    Returns: X_aligned (time t), y_aligned (time t+1)
    """
    # Reset index to work with dates
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column for clarity
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Create a shifted version of y for each stock
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift y backwards by 1 year (so y at t-1 pairs with X at t)
        # Or more intuitively: align X at year t with y at year t+1
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']], 
            on=['order_book_id', 'date']
        )
        
        # Drop rows where next year's target is missing
        merged = merged.dropna(subset=['target_next_year'])
        
        aligned_pairs.append(merged)
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X and y
    X_aligned = aligned_df.drop(['order_book_id', 'date', 'target_next_year'], axis=1)
    y_aligned = aligned_df['target_next_year']
    
    return X_aligned, y_aligned, aligned_df[['order_book_id', 'date']]  # Keep metadata

def robust_align_features_target(X, y):
    """
    Robust alignment handling index mismatches
    """
    # Reset indices to columns
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Sort both by stock and date
    X_df = X_df.sort_values(['order_book_id', 'date'])
    y_df = y_df.sort_values(['order_book_id', 'date'])
    
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        if len(y_stock) == 0:
            print(f"Warning: No target data for {stock}")
            continue
        
        # Ensure both have consistent date ranges
        common_dates = set(X_stock['date']).intersection(set(y_stock['date']))
        if len(common_dates) == 0:
            print(f"Warning: No common dates for {stock}")
            continue
        
        X_stock = X_stock[X_stock['date'].isin(common_dates)]
        y_stock = y_stock[y_stock['date'].isin(common_dates)]
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift target to next year
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']],
            on=['order_book_id', 'date'],
            how='inner'
        )
        
        # Drop rows with NaN target (last year of each stock)
        merged = merged.dropna(subset=['target_next_year'])
        
        if len(merged) > 0:
            aligned_pairs.append(merged)
    
    if len(aligned_pairs) == 0:
        raise ValueError("No aligned data found. Check your date ranges.")
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X, y, and metadata
    feature_cols = [col for col in X_df.columns if col not in ['order_book_id', 'date']]
    X_aligned = aligned_df[feature_cols]
    y_aligned = aligned_df['target_next_year']
    metadata = aligned_df[['order_book_id', 'date']]
    
    return X_aligned, y_aligned, metadata

WINDOW_IN = 5
WINDOW_OUT = 5

def create_5y_sequences(X, y):

    X_df = X.reset_index().sort_values(
        ['order_book_id', 'date']
    )

    y_df = y.reset_index().sort_values(
        ['order_book_id', 'date']
    )

    target_col = y_df.columns[-1]

    X_seq = []
    y_seq = []

    feature_cols = [
        c for c in X_df.columns
        if c not in ['order_book_id', 'date']
    ]

    for stock in X_df['order_book_id'].unique():

        xf = X_df[X_df['order_book_id'] == stock]
        yf = y_df[y_df['order_book_id'] == stock]

        xvals = xf[feature_cols].values
        yvals = yf[target_col].values

        if len(xvals) < WINDOW_IN + WINDOW_OUT:
            continue

        max_start = min(
            len(xvals),
            len(yvals)
        ) - WINDOW_IN - WINDOW_OUT + 1

        for i in range(max_start):

            X_seq.append(
                xvals[i:i + WINDOW_IN].flatten()
            )

            y_seq.append(
                yvals[
                    i + WINDOW_IN :
                    i + WINDOW_IN + WINDOW_OUT
                ]
            )

    # print("Number of samples:", len(y_seq))

    # for i in range(10):
    #     print(i, np.array(y_seq[i]).shape)
    
    # unique_shapes = set()

    # for item in y_seq:
    #     unique_shapes.add(np.array(item).shape)

    # print(unique_shapes)
    
    return np.array(X_seq), np.array(y_seq)

# Apply the robust alignment
try:
    # X_aligned, y_aligned, metadata = robust_align_features_target(X, y)
    X_aligned, y_aligned = create_5y_sequences(X, y)
    print(f"Success! Aligned {len(X_aligned)} samples")
    print(f"X shape: {X_aligned.shape}")
    print(f"y shape: {y_aligned.shape}")
    # print(f"Date range: {metadata['date'].min()} to {metadata['date'].max()}")
except ValueError as e:
    print(f"Alignment failed: {e}")

# # Apply alignment
# X_aligned, y_aligned, metadata = align_features_target(X, y)

# print(f"Original X shape: {X.shape}")
# print(f"Original y shape: {y.shape}")
# print(f"Aligned X shape: {X_aligned.shape}")
# print(f"Aligned y shape: {y_aligned.shape}")

Success! Aligned 447 samples
X shape: (447, 210)
y shape: (447, 5)


In [40]:
print(X_aligned)

[[1.31596390e+10 1.31596390e+10 0.00000000e+00 ... 2.14435800e+12
  1.26736000e+11 1.26736000e+11]
 [2.07014010e+10 2.07014010e+10 0.00000000e+00 ... 2.59906000e+12
  1.57136000e+11 1.57136000e+11]
 [2.23804299e+10 2.23804299e+10 0.00000000e+00 ... 5.19974136e+11
  7.85294850e+10 1.06182513e+11]
 ...
 [1.04852300e+09 1.04852300e+09 0.00000000e+00 ... 1.15059188e+10
  1.04775996e+10 1.04775996e+10]
 [4.43565143e+09 4.43565143e+09 0.00000000e+00 ... 4.47894934e+10
  1.78334946e+10 1.82817487e+10]
 [3.99369138e+09 3.99369138e+09 0.00000000e+00 ... 1.00207157e+11
  3.72785081e+10 4.48829957e+10]]


In [41]:
print(y_aligned)

[[ 2.15918668  2.22861137  1.98062002  3.05200709  1.53360432]
 [ 2.22861137  1.98062002  3.05200709  1.53360432  2.0025592 ]
 [ 4.93004984  2.21270379  0.65814586  1.33431067 -0.87460302]
 ...
 [ 2.42778615  3.77199916  0.62062529  2.64492847  2.16288811]
 [ 0.23018945  0.33071197  0.29191985  0.29884922  0.67539088]
 [ 0.33071197  0.29191985  0.29884922  0.67539088  1.04383936]]


## 随机森林

In [42]:
# 随机森林模型

# Train test split 80/20
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Define hyperparameters once
rf_params = dict(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=5,
    max_features='sqrt'
)

# Feature selection
model = RandomForestRegressor(**rf_params)
selector = RFECV(estimator=model, cv=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)  # apply same selection to test set

# Cross-validate, then final fit
rf_model = RandomForestRegressor(**rf_params)
cv_scores = cross_val_score(rf_model, X_train_selected, y_train, cv=5, scoring='r2')
print(f"Cross-validation R^2 scores: {cv_scores}")

# Fit the model
rf_model.fit(X_train_selected, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_selected)
r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
mae = mean_absolute_error(y_test, y_pred)
print(f"Test Average R^2: {r2}, Test MAE: {mae}")

r2_per_year = r2_score(
    y_test,
    y_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

Cross-validation R^2 scores: [ 0.37096666 -0.24404015 -0.10046758  0.13105914  0.18695182]
Test Average R^2: 0.09531193565319127, Test MAE: 0.9052704964892098
Year+1 R^2: 0.5691867401841915
Year+2 R^2: 0.13295248779220292
Year+3 R^2: -0.5658021915562172
Year+4: R^2 -0.06133737665550032
Year+5 R^2: 0.40156001850127954


## Gradient Boosting Regression Model

In [43]:
# Grading Boosting Regression Model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import MultiOutputRegressor

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Gradient Boosting Regressor model
gbr_model = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    )
)

# Cross validation
gbr_cv_scores = cross_val_score(
    gbr_model,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("Gradient Boosting Cross-validation R^2 scores:")
print(gbr_cv_scores)
print(f"Average CV R^2: {gbr_cv_scores.mean()}")

# Train model
gbr_model.fit(X_train, y_train)

# Prediction
gbr_pred = gbr_model.predict(X_test)

# Evaluation
gbr_r2 = r2_score(y_test, gbr_pred)
gbr_mae = mean_absolute_error(y_test, gbr_pred)

print(f"Gradient Boosting Test R^2: {gbr_r2}")
print(f"Gradient Boosting Test MAE: {gbr_mae}")

r2_per_year = r2_score(
    y_test,
    gbr_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

Gradient Boosting Cross-validation R^2 scores:
[ 0.44293058 -0.00682272 -0.91117052  0.17685732  0.2747465 ]
Average CV R^2: -0.004691769355802622
Gradient Boosting Test R^2: -0.1130744816940471
Gradient Boosting Test MAE: 1.0570908048214314
Year+1 R^2: 0.08488045595128013
Year+2 R^2: 0.032665797089972304
Year+3 R^2: -0.4129760299985623
Year+4: R^2 -0.1288918715640519
Year+5 R^2: -0.14105075994887373


## 长短期记忆

In [44]:
# 长短期记忆模型

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Scale features
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

# Reshape for LSTM
# Shape: (samples, timesteps, features)
n_features = X_aligned.shape[1] // 5

X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    5,
    n_features
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    5,
    n_features
)

# Build LSTM model
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        units=64,
        return_sequences=False,
        input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])
    )
)

lstm_model.add(Dropout(0.2))

lstm_model.add(Dense(32, activation='relu'))

lstm_model.add(Dense(5))

# Compile
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)

# Train
history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
lstm_pred_scaled = lstm_model.predict(X_test_lstm)

# Inverse transform
lstm_pred = y_scaler.inverse_transform(lstm_pred_scaled)

# Evaluation
lstm_r2 = r2_score(y_test, lstm_pred, multioutput='uniform_average')
lstm_mae = mean_absolute_error(y_test, lstm_pred)

print(f"LSTM Test Average R^2: {lstm_r2}")
print(f"LSTM Test MAE: {lstm_mae}")

r2_per_year = r2_score(
    y_test,
    lstm_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

Epoch 1/50
9/9 [==============================] - 4s 89ms/step - loss: 0.0134 - val_loss: 0.0027
Epoch 2/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0075 - val_loss: 0.0012
Epoch 3/50
9/9 [==============================] - 0s 15ms/step - loss: 0.0063 - val_loss: 9.0367e-04
Epoch 4/50
9/9 [==============================] - 0s 12ms/step - loss: 0.0057 - val_loss: 8.7711e-04
Epoch 5/50
9/9 [==============================] - 0s 13ms/step - loss: 0.0046 - val_loss: 7.7801e-04
Epoch 6/50
9/9 [==============================] - 0s 13ms/step - loss: 0.0041 - val_loss: 9.3835e-04
Epoch 7/50
9/9 [==============================] - 0s 14ms/step - loss: 0.0036 - val_loss: 0.0010
Epoch 8/50
9/9 [==============================] - 0s 14ms/step - loss: 0.0039 - val_loss: 9.3870e-04
Epoch 9/50
9/9 [==============================] - 0s 13ms/step - loss: 0.0029 - val_loss: 0.0013
Epoch 10/50
9/9 [==============================] - 0s 23ms/step - loss: 0.0028 - val_loss: 0.0014
Epoch 11/

## 预测

In [45]:
# 预测

# Fetch financial statement ratio information
ly_financial_df = rq.get_factor('601318.XSHG', factors, start_date='2022-01-01', end_date='2026-05-01')

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

ly_financial_df = ly_financial_df.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

ly_financial_df = ly_financial_df.reorder_levels(['order_book_id', 'date'])

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

# Keep only latest 5 years
ly_financial_df = ly_financial_df.tail(5)

print(ly_financial_df.head(20))

                               revenue  operating_revenue  \
order_book_id date                                          
601318.XSHG   2022-12-31  8.702020e+11       8.702020e+11   
              2023-12-31  7.049380e+11       7.049380e+11   
              2024-12-31  7.753830e+11       7.753830e+11   
              2025-12-31  8.329400e+11       8.329400e+11   
              2026-12-31  2.184050e+11       2.184050e+11   

                          net_operating_revenue  book_value_per_share  \
order_book_id date                                                      
601318.XSHG   2022-12-31                    0.0               45.8862   
              2023-12-31                    0.0               49.5938   
              2024-12-31                    0.0               49.8594   
              2025-12-31                    0.0               54.4735   
              2026-12-31                    0.0               56.2354   

                          book_value_per_share_lf  book_valu

In [46]:
prediction_input = (
    ly_financial_df
    .values
    .flatten()
    .reshape(1, -1)
)

In [47]:
# Random Forest Prediction
rf_prediction = rf_model.predict(
    selector.transform(prediction_input)
)

# Gradient Boosting Regressor Prediction
gbr_prediction = gbr_model.predict(
    prediction_input
)

# LSTM Prediction

# Scale
ly_scaled = x_scaler.transform(prediction_input)

# Reshape
n_features = ly_scaled.shape[1] // 5
ly_lstm = ly_scaled.reshape(
    ly_scaled.shape[0],
    5,
    n_features
)

# Predict
lstm_prediction_scaled = lstm_model.predict(ly_lstm)

# Inverse scale
lstm_prediction = y_scaler.inverse_transform(
    lstm_prediction_scaled
)

print(f"Random Forest Prediction: {rf_prediction}")
print(f"Gradient Boosting Regressor Prediction: {gbr_prediction}")
print(f"LSTM Prediction: {lstm_prediction}")

1/1 [==============================] - 0s 37ms/step
Random Forest Prediction: [[ 6.7070339   9.51703068 11.42260751 12.882441   13.18482302]]
Gradient Boosting Regressor Prediction: [[27.82602502 25.44647333 30.89664924 35.21398505 32.36398927]]
LSTM Prediction: [[-3.9262185 -4.851963  -8.568272   4.244317   6.554291 ]]


## 模型结果

In [48]:
# =========================
# Model Comparison
# =========================

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting Regression', 'LSTM'],
    'R2': [
        r2,
        gbr_r2,
        lstm_r2
    ],
    'MAE': [
        mae,
        gbr_mae,
        lstm_mae
    ]
})

print(comparison_df)

                          Model         R2       MAE
0                 Random Forest   0.095312  0.905270
1  Gradient Boosting Regression  -0.113074  1.057091
2                          LSTM -10.860127  2.218065


## DCF

In [49]:
import numpy as np

def terminal_value_gordon(cf_final_year, discount_rate, growth_rate):
    return cf_final_year * (1 + growth_rate) / (discount_rate - growth_rate)

def dcf_valuation(cash_flows, discount_rate, terminal_value=None):
    """
    Calculate DCF valuation.

    Parameters:
    - cash_flows: list or array of future cash flows [CF1, CF2, ...]
    - discount_rate: required return (e.g. 0.1 for 10%)
    - terminal_value: optional lump sum value at final year

    Returns:
    - present value (DCF valuation)
    """
    cash_flows = np.array(cash_flows, dtype=float)

    # Discount each cash flow
    years = np.arange(1, len(cash_flows) + 1)
    discounted_cf = cash_flows / (1 + discount_rate) ** years

    pv = discounted_cf.sum()

    # Add terminal value if provided
    if terminal_value is not None:
        pv_terminal = terminal_value / (1 + discount_rate) ** len(cash_flows)
        pv += pv_terminal

    return pv

# Assume 15% growth rate and 20% discount rate
growth_rate = 0.15 # 15%
discount_rate = 0.20  # 20%

rf_dcf = dcf_valuation(rf_prediction, discount_rate, terminal_value_gordon(rf_prediction[0][-1], discount_rate, growth_rate))
gbr_dcf = dcf_valuation(gbr_prediction, discount_rate, terminal_value_gordon(gbr_prediction[0][-1], discount_rate, growth_rate))
lstm_dcf = dcf_valuation(lstm_prediction, discount_rate, terminal_value_gordon(lstm_prediction[0][-1], discount_rate, growth_rate))
print("Random Forect DCF Value:", rf_dcf)
print("Gradient Boosting Regressor DCF Value:", gbr_dcf)
print("LSTM DCF Value:", lstm_dcf)

Random Forect DCF Value: 297.4707213373212
Gradient Boosting Regressor DCF Value: 746.7657292525023
LSTM DCF Value: 120.16736865043636


## Export Models

In [50]:
import joblib
import pickle
from tensorflow.keras.models import save_model

# =========================
# Export Models
# =========================

# 1. Export Random Forest model and its selector
joblib.dump(rf_model, 'ffcf_random_forest_model.pkl')
joblib.dump(selector, 'ffcf_rf_selector.pkl')
print("✓ Random Forest model and selector saved")

# 2. Export Gradient Boosting model
joblib.dump(gbr_model, 'ffcf_gradient_boosting_model.pkl')
print("✓ Gradient Boosting model saved")

# 3. Export LSTM model
lstm_model.save('ffcf_lstm_model.h5')
print("✓ LSTM model saved")

# 4. Export scalers (needed for LSTM preprocessing)
joblib.dump(x_scaler, 'ffcf_x_scaler.pkl')
joblib.dump(y_scaler, 'ffcf_y_scaler.pkl')
print("✓ Scalers saved")

# 5. Save feature columns and configuration
config = {
    'window_in': 5,
    'window_out': 5,
    'n_features': n_features,
    'factors': factors if 'factors' in dir() else None
}

with open('ffcf_model_config.pkl', 'wb') as f:
    pickle.dump(config, f)
print("✓ Model configuration saved")

print("\n✅ All models exported successfully!")

✓ Random Forest model and selector saved
✓ Gradient Boosting model saved
✓ LSTM model saved
✓ Scalers saved
✓ Model configuration saved

✅ All models exported successfully!


/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
